
# PhishU — Master Notebook (EDA • PCA • Tabular Models • Semantic Baselines)

This notebook orchestrates all analyses and models without changing individual scripts.

**Sections**  
1. Environment & Logging  
2. Data loading (single source of truth)  
3. EDA (src/data_analysis/EDA.py)  
4. PCA / ACP (src/data_analysis/ACP.py)  
5. Tabular models: Logistic Regression, Random Forest, XGBoost (src/models/*.py)  
6. Semantic baselines: TF-IDF+LinearSVC, FastText+LogReg (src/semantic_models/...)  
7. Summary comparison table


## 1) Environment & Logging

In [2]:
from src.pipeline.logger import init_logging, get_logger, set_seed
init_logging("INFO")  # set "DEBUG" for more verbose logs
log = get_logger("Notebook", "INFO")
set_seed(42)
log.info("Notebook started.")


[13:10:46] [INFO] Notebook: Notebook started.


## 2) Data loading

In [3]:

# Load from UCI via our shared DataLoader
from src.utils.data_loader import DataLoader

dl = DataLoader(dataset_id=967)  # PhiUSIIL Phishing URL dataset
X_all, y_series, meta = dl.get_xy_as_dataframes()

# Build a single DataFrame
df = X_all.copy()
df["label"] = y_series.astype(int).values  # ensure numeric labels

log.info(f"Loaded from UCI: {df.shape[0]} rows, {df.shape[1]} cols "
         f"(features={X_all.shape[1]})")
df.head(3)

[13:18:49] [INFO] Notebook: Loaded from UCI: 235795 rows, 55 cols (features=54)


,URL,URLLength,Domain,DomainLength,IsDomainIP,TLD,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,...,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef,label
0,https://www.southbankmosaics.com,31,www.southbankmosaics.com,24,0,com,100.0,1.000000,0.522907,0.061933,...,0,0,1,34,20,28,119,0,124,1
1,https://www.uni-mainz.de,23,www.uni-mainz.de,16,0,de,100.0,0.666667,0.032650,0.050207,...,0,0,1,50,9,8,39,0,217,1
2,https://www.voicefmradio.co.uk,29,www.voicefmradio.co.uk,22,0,uk,100.0,0.866667,0.028555,0.064129,...,0,0,1,10,2,7,42,2,5,1


## 3) EDA

In [4]:

import pandas as pd
from src.data_analysis.EDA import run_eda

eda_out = run_eda(
    df=df,
    label_col="label",
    save_dir="outputs/eda",
    save_fig=True,
    show_fig=False
)

display(pd.Series(eda_out["shape"], index=["rows","cols"]).to_frame("shape"))
display(eda_out["dtypes_counts"].to_frame("count").T)
display(eda_out["label_distribution_pct"].to_frame("pct"))
if eda_out["preview_describe"].shape[0] > 0:
    display(eda_out["preview_describe"])


[13:19:27] [INFO] EDA: ▶ Describing preview columns (5 cols) ...


[13:19:27] [INFO] EDA: ✓ Describing preview columns (5 cols) done in 0.1s
[13:19:27] [INFO] EDA: ▶ Creating label distribution plot ...
[13:19:28] [INFO] EDA: Figure saved to: outputs/eda\distribution_label.png
[13:19:28] [INFO] EDA: ✓ Creating label distribution plot done in 0.2s


,shape
rows,235795
cols,55


,int64,float64,object
count,41,10,4


,pct
label,
1,57.189508
0,42.810492


,URLLength,DomainLength,NoOfJS,NoOfImage,NoOfExternalRef
count,235795.000000,235795.000000,235795.000000,235795.000000,235795.000000
mean,34.573095,21.470396,10.522305,26.075689,49.262516
std,41.314153,9.150793,22.312192,79.411815,161.027430
min,13.000000,4.000000,0.000000,0.000000,0.000000
25%,23.000000,16.000000,0.000000,0.000000,1.000000
50%,27.000000,20.000000,6.000000,8.000000,10.000000
75%,34.000000,24.000000,15.000000,29.000000,57.000000
max,6097.000000,110.000000,6957.000000,8956.000000,27516.000000


## 4) PCA / ACP

In [5]:

from src.data_analysis.ACP import run_pca

pca_out = run_pca(
    df=df,
    label_col="label",
    n_components=0.95,
    save_dir="outputs/pca",
    save_fig=True,
    show_fig=False
)

log.info(f"PCA retained components: {pca_out['n_components_']} "
         f"(cumulative variance={pca_out['explained_variance_ratio'].sum():.3f})")
display(pca_out["components_df"].head(5))
display(pd.DataFrame({
    "PC1_top": pca_out["pc1_top_loadings"],
    "PC2_top": pca_out["pc2_top_loadings"]
}))


[13:19:55] [INFO] ACP: ▶ Selecting numeric columns ...
[13:19:55] [INFO] ACP: ✓ Selecting numeric columns done in 0.1s
[13:19:55] [INFO] ACP: ▶ Scaling features (StandardScaler) ...
[13:19:55] [INFO] ACP: ✓ Scaling features (StandardScaler) done in 0.2s
[13:19:55] [INFO] ACP: ▶ Fitting PCA (n_components=0.95) ...
[13:19:55] [INFO] ACP: ✓ Fitting PCA (n_components=0.95) done in 0.1s
[13:19:55] [INFO] ACP: Retained components: 36
[13:19:55] [INFO] ACP: Cumulative explained variance: 0.9507
[13:19:55] [INFO] ACP: ▶ Building components (loadings) DataFrame ...
[13:19:55] [INFO] ACP: ✓ Building components (loadings) DataFrame done in 0.0s
[13:19:55] [INFO] ACP: ▶ Creating Scree plot (cumulative explained variance) ...
[13:19:55] [INFO] ACP: ✓ Creating Scree plot (cumulative explained variance) done in 0.3s
[13:19:55] [INFO] ACP: ▶ Creating 2D scatter on first two PCs ...
[13:20:05] [INFO] ACP: ✓ Creating 2D scatter on first two PCs done in 9.8s
[13:20:05] [INFO] Notebook: PCA retained compo

,URLLength,DomainLength,IsDomainIP,URLSimilarityIndex,CharContinuationRate,TLDLegitimateProb,URLCharProb,TLDLength,NoOfSubDomain,HasObfuscation,...,Bank,Pay,Crypto,HasCopyrightInfo,NoOfImage,NoOfCSS,NoOfJS,NoOfSelfRef,NoOfEmptyRef,NoOfExternalRef
PC1,-0.170095,-0.134718,-0.058771,0.286119,0.207192,0.072324,0.182087,0.000095,-0.068942,-0.044377,...,0.084618,0.134157,0.044735,0.224363,0.111093,0.029155,0.133114,0.125423,0.045029,0.104293
PC2,0.324775,0.020200,0.125017,0.017056,0.008703,0.026520,-0.005669,-0.000445,0.009367,0.130631,...,0.081272,0.106953,0.031041,0.136251,0.078345,0.022841,0.085552,0.086069,0.035035,0.072569
PC3,0.028841,-0.231840,-0.025855,0.085568,0.318810,0.193648,0.117737,0.162269,-0.309943,0.091201,...,-0.139955,-0.133901,-0.037968,-0.066753,-0.140806,-0.049191,-0.097251,-0.168148,-0.069285,-0.142677
PC4,0.160260,0.256281,0.035134,-0.130614,0.004626,0.366830,0.109629,0.363739,-0.173579,-0.253843,...,0.087275,0.087231,0.026008,-0.022263,0.005337,-0.002240,-0.003534,-0.048682,0.008108,-0.041230
PC5,-0.056261,-0.005228,-0.116075,-0.075127,0.062607,0.107745,-0.014844,0.097087,-0.092080,0.203110,...,0.188448,0.117396,0.083099,-0.119841,0.279052,0.106518,0.003368,0.298222,0.112660,0.268772


,PC1_top,PC2_top
CharContinuationRate,0.207192,NaN
DomainTitleMatchScore,0.230340,NaN
HasCopyrightInfo,0.224363,NaN
HasDescription,0.216431,NaN
HasSocialNet,0.241459,NaN
HasSubmitButton,0.188332,0.147001
IsHTTPS,NaN,0.152316
NoOfAmpersandInURL,NaN,0.308973
NoOfDegitsInURL,NaN,0.343919
NoOfEqualsInURL,NaN,0.350142


## 5) Tabular models

In [7]:

from src.models.regressionlogistique import run_logistic_regression
from src.models.randomforest import run_random_forest
from src.models.XGboost import run_xgboost

tabular_results = {}
common_features = (
    'URLLength','DomainLength','NoOfSubDomain','IsDomainIP',
    'NoOfLettersInURL','NoOfDegitsInURL','NoOfEqualsInURL',
    'NoOfQMarkInURL','NoOfAmpersandInURL','NoOfOtherSpecialCharsInURL',
    'SpacialCharRatioInURL','TLDLength'
)

log.info("Running Logistic Regression (tabular)...")
logreg_out = run_logistic_regression(
    df=df,
    label_col="label",
    features=common_features,
    sample_n=10000,
    save_dir="outputs/logreg",
    save_fig=True,
    show_fig=False
)
tabular_results["LogisticRegression"] = logreg_out["metrics"]
display(pd.Series(logreg_out["metrics"], name="LogisticRegression"))

log.info("Running Random Forest (tabular)...")
rf_out = run_random_forest(
    df=df,
    label_col="label",
    features=logreg_out["X_columns"],
    sample_n=10000,
    save_dir="outputs/random_forest",
    save_fig=True,
    show_fig=False
)
tabular_results["RandomForest"] = rf_out["metrics"]
display(pd.Series(rf_out["metrics"], name="RandomForest"))

log.info("Running XGBoost (tabular)...")
xgb_out = run_xgboost(
    df=df,
    label_col="label",
    features=logreg_out["X_columns"],
    sample_n=10000,
    save_dir="outputs/xgboost",
    save_fig=True,
    show_fig=False
)
tabular_results["XGBoost"] = xgb_out["metrics"]
display(pd.Series(xgb_out["metrics"], name="XGBoost"))


[13:23:05] [INFO] Notebook: Running Logistic Regression (tabular)...
[13:23:05] [INFO] LogRegTab: ▶ Stratified sampling to n=10000 ...
[13:23:05] [INFO] LogRegTab: ✓ Stratified sampling to n=10000 done in 0.2s
[13:23:05] [INFO] LogRegTab: Échantillon sélectionné : 10000 lignes
[13:23:05] [INFO] LogRegTab: Variables conservées (12 au total) : ['URLLength', 'DomainLength', 'NoOfSubDomain', 'IsDomainIP', 'NoOfLettersInURL', 'NoOfDegitsInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'TLDLength']
[13:23:05] [INFO] LogRegTab: ▶ Scaling features (StandardScaler) ...
[13:23:05] [INFO] LogRegTab: ✓ Scaling features (StandardScaler) done in 0.0s
[13:23:05] [INFO] LogRegTab: ▶ Training LogisticRegression ...
[13:23:05] [INFO] LogRegTab: ✓ Training LogisticRegression done in 0.0s
[13:23:05] [INFO] LogRegTab: ▶ Scoring ...
[13:23:05] [INFO] LogRegTab: ✓ Scoring done in 0.0s
[13:23:05] [INFO] LogRegTab: Modèle basé sur 10 000 

accuracy     0.860500
precision    0.813633
recall       0.980769
f1           0.889417
roc_auc      0.938074
Name: LogisticRegression, dtype: float64

[13:23:06] [INFO] Notebook: Running Random Forest (tabular)...
[13:23:06] [INFO] RandomForestTab: ▶ Stratified sampling to n=10000 ...
[13:23:06] [INFO] RandomForestTab: ✓ Stratified sampling to n=10000 done in 0.2s
[13:23:06] [INFO] RandomForestTab: Using 12 features: ['URLLength', 'DomainLength', 'NoOfSubDomain', 'IsDomainIP', 'NoOfLettersInURL', 'NoOfDegitsInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'TLDLength']
[13:23:06] [INFO] RandomForestTab: ▶ Scaling features (StandardScaler) ...
[13:23:06] [INFO] RandomForestTab: ✓ Scaling features (StandardScaler) done in 0.0s
[13:23:06] [INFO] RandomForestTab: ▶ Training RandomForest ...
[13:23:06] [INFO] RandomForestTab: ✓ Training RandomForest done in 0.4s
[13:23:06] [INFO] RandomForestTab: ▶ Scoring ...
[13:23:06] [INFO] RandomForestTab: ✓ Scoring done in 0.1s
[13:23:06] [INFO] RandomForestTab: Acc=0.8785 | Prec=0.8390 | Rec=0.9747 | F1=0.9017 | AUC=0.9734
[13:

accuracy     0.878500
precision    0.838977
recall       0.974650
f1           0.901739
roc_auc      0.973436
Name: RandomForest, dtype: float64

[13:23:07] [INFO] Notebook: Running XGBoost (tabular)...
[13:23:07] [INFO] XGBoostTab: ▶ Stratified sampling to n=10000 ...
[13:23:07] [INFO] XGBoostTab: ✓ Stratified sampling to n=10000 done in 0.2s
[13:23:07] [INFO] XGBoostTab: Using 12 features: ['URLLength', 'DomainLength', 'NoOfSubDomain', 'IsDomainIP', 'NoOfLettersInURL', 'NoOfDegitsInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'TLDLength']
[13:23:07] [INFO] XGBoostTab: ▶ Scaling features (StandardScaler) ...
[13:23:07] [INFO] XGBoostTab: ✓ Scaling features (StandardScaler) done in 0.0s
[13:23:07] [INFO] XGBoostTab: ▶ Training XGBClassifier ...
[13:23:09] [INFO] XGBoostTab: ✓ Training XGBClassifier done in 1.7s
[13:23:09] [INFO] XGBoostTab: ▶ Scoring ...
[13:23:09] [INFO] XGBoostTab: ✓ Scoring done in 0.0s
[13:23:09] [INFO] XGBoostTab: Acc=0.9830 | Prec=0.9826 | Rec=0.9878 | F1=0.9852 | AUC=0.9962
[13:23:09] [INFO] XGBoostTab: ▶ Rendering confusion matrix

c:\Users\frohl\Documents\CentraleSupélec\3A\DAML\Projet\.env\Lib\site-packages\xgboost\training.py:199: UserWarning: [13:23:08] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


[13:23:09] [INFO] XGBoostTab: ✓ Rendering confusion matrix done in 0.2s
[13:23:09] [INFO] XGBoostTab: ▶ Rendering feature importances ...
[13:23:09] [INFO] XGBoostTab: ✓ Rendering feature importances done in 0.3s


accuracy     0.983000
precision    0.982609
recall       0.987762
f1           0.985179
roc_auc      0.996236
Name: XGBoost, dtype: float64

## 6) Semantic baselines (TF-IDF+LinearSVC, FastText+LogReg)

In [8]:

from src.semantic_models.phishing_url_semantic_baselines import (
    train_and_compare_semantic_baselines, PhiUSIILSpec
)

sem_out = train_and_compare_semantic_baselines(
    spec=PhiUSIILSpec(dataset_id=967),
    test_size=0.2,
    random_state=42,
    run_char_tfidf=True,
    run_fasttext=True,
    fasttext_fast_mode=True
)

def _extract_metrics(res_dict):
    out = {}
    for k, v in res_dict.items():
        out[k] = {
            "roc_auc": float(v.get("roc_auc", float("nan"))),
            "pr_auc": float(v.get("pr_auc", float("nan")))
        }
    return out

sem_metrics = _extract_metrics(sem_out)
sem_metrics


[13:23:34] [INFO] Semantic: [Data] Loading URLs & labels
[13:24:14] [INFO] Semantic: [Data] Train/test split
[13:24:14] [INFO] Semantic: [Data] Train=188636 | Test=47159
[13:24:14] [INFO] Semantic: [TF-IDF+SVC] Cleaning & vectorizing


Cleaning URLs (char-ngrams):   0%|          | 0/188636 [00:00<?, ?it/s]

Cleaning URLs (char-ngrams):   0%|          | 0/47159 [00:00<?, ?it/s]

[13:24:31] [INFO] Semantic: [TF-IDF+SVC] Training LinearSVC
[13:24:35] [INFO] Semantic: [TF-IDF+SVC] Evaluating

=== TF-IDF char (3–5) + LinearSVC ===
ROC-AUC: 0.9424
PR-AUC : 0.9390

-- Classification report --
              precision    recall  f1-score   support

           0      0.944     0.822     0.879     20189
           1      0.879     0.964     0.919     26970

    accuracy                          0.903     47159
   macro avg      0.911     0.893     0.899     47159
weighted avg      0.907     0.903     0.902     47159

-- Confusion matrix --
[[16595  3594]
 [  978 25992]]
[13:24:35] [INFO] Semantic: [FastText+LR] Tokenizing train/test URLs


Tokenizing URLs (word tokens):   0%|          | 0/188636 [00:00<?, ?it/s]

...

[13:25:28] [INFO] Semantic: [FastText+LR] Embedding FULL train/test sets


Averaging embeddings:   0%|          | 0/188636 [00:00<?, ?it/s]

Averaging embeddings:   0%|          | 0/47159 [00:00<?, ?it/s]

[13:25:41] [INFO] Semantic: [FastText+LR] Training LogisticRegression
[13:25:42] [INFO] Semantic: [FastText+LR] Evaluating

=== FastText (moyenne) + LogisticRegression ===
ROC-AUC: 0.9237
PR-AUC : 0.9199

-- Classification report --
              precision    recall  f1-score   support

           0      0.902     0.790     0.842     20189
           1      0.856     0.936     0.894     26970

    accuracy                          0.873     47159
   macro avg      0.879     0.863     0.868     47159
weighted avg      0.876     0.873     0.872     47159

-- Confusion matrix --
[[15946  4243]
 [ 1727 25243]]


{'char_tfidf_svc': {'roc_auc': 0.9424456590080985,
  'pr_auc': 0.9390291999323371},
 'fasttext_logreg': {'roc_auc': 0.9237278592348652,
  'pr_auc': 0.9199168928574093}}

## 7) Summary comparison table

In [9]:

import pandas as pd
tab_df = pd.DataFrame(tabular_results).T[["accuracy","precision","recall","f1","roc_auc"]]
sem_df = pd.DataFrame(sem_metrics).T[["roc_auc","pr_auc"]]

display(tab_df.style.format("{:.4f}").set_caption("Tabular models"))
display(sem_df.style.format("{:.4f}").set_caption("Semantic baselines"))


,accuracy,precision,recall,f1,roc_auc
LogisticRegression,0.8605,0.8136,0.9808,0.8894,0.9381
RandomForest,0.8785,0.8390,0.9747,0.9017,0.9734
XGBoost,0.9830,0.9826,0.9878,0.9852,0.9962


,roc_auc,pr_auc
char_tfidf_svc,0.9424,0.9390
fasttext_logreg,0.9237,0.9199
